<a href="https://colab.research.google.com/github/ajtamayoh/In2Lab-TNT-at-SMM4H-2026/blob/main/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In2Lab-TNT at \#SMM4H-HeaRD 2026: An Application of QTT's Terminological Entanglement to Leverage Insomnia Detection in Clinical Notes

# 1. Library Installation and Configuration

First, we install the OpenAI SDK and prepare the credentials.

In [ ]:
!pip install openai pandas scikit-learn

import pandas as pd
import json
import os
from openai import OpenAI
from sklearn.metrics import f1_score, recall_score, precision_score, classification_report

# OpenAI API Key configuration
os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()

# 2. Data Loading and Preparation

This block reads the CSV and JSON files and merges them into a single Pandas DataFrame using the note_id.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Load the corpus (CSV)

#Training data
#df_corpus = pd.read_csv('/training/corpus_training.csv') # columnas: note_id, text

# 2. Loading the labels (JSON) for training
#with open('/training/subtask_1.json', 'r') as f:
#    labels_dict = json.load(f)

#Validation data
df_corpus = pd.read_csv('/validation/corpus_validation.csv') # columnas: note_id, text

# 2. Loading the labels (JSON) for validation
with open('/validation/subtask_1.json', 'r') as f:
    labels_dict = json.load(f)

# JSON to plane format (DataFrame)
labels_list = []
for note_id, val in labels_dict.items():
    labels_list.append({
        'note_id': int(note_id),
        'label': val['Insomnia'].lower() # 'yes'/'no'
    })
df_labels = pd.DataFrame(labels_list)

# 3. Mergear ambos archivos por 'note_id'
df_final = pd.merge(df_corpus, df_labels, on='note_id')

print(f"Total de registros cargados: {len(df_final)}")
df_final.head()

Total de registros cargados: 23


,note_id,text,label
0,10686,male patient in forties prescribed no drugs\n\...,no
1,1262007,female patient in forties prescribed Iso-Osmot...,yes
2,1272946,"female patient in sixties prescribed D5W, D5 1...",yes
3,1279443,female patient in eighties prescribed no drugs...,yes
4,1281746,male patient in fifties prescribed Iso-Osmotic...,no


# 3. Classification Function with GPT-4o Mini

We define the logic used to query the model. We employ a strict System Prompt so that the model responds only with "yes" or "no", which facilitates downstream processing.

In [ ]:
def classify_insomnia(text):
    """
    self-check prompt engineering strategy
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": """You are a medical classifier. Respond ONLY with the word 'yes' if the text indicates insomnia, or 'no' if it does not. Do not explain. The following are the rules to decide your response:

                **Definition 1: Difficulty Sleeping**
                  The patient is considered to have difficulty sleeping if they report any of the following:
                  1. Trouble initiating sleep.
                  2. Trouble maintaining sleep.
                  3. Waking up earlier than desired.
                  4. An explicit mention of insomnia.

                  **Definition 2: Daytime Impairment**
                  The patient is considered to have daytime impairment if they report any of the following:
                  1. Fatigue or malaise.
                  2. Impaired attention, concentration, or memory.
                  3. Impaired social, family, occupational, or academic performance.
                  4. Mood disturbance or irritability.
                  5. Daytime sleepiness.
                  6. Behavioral problems such as hyperactivity, impulsivity, or aggression.
                  7. Decreased motivation, energy, or initiative.
                  8. Proneness to errors or accidents.
                  9. Concerns or dissatisfaction with sleep.
                  10. An explicit mention of insomnia.

                  **Rule A:** The patient is considered to have insomnia if they meet both Definition 1 (difficulty sleeping) and Definition 2 (daytime impairment).

                  **Rule B**: The patient has insomnia if prescribed any of the following primary insomnia medications: Estazolam, Eszopiclone, Flurazepam, Lemborexant, Quazepam, Ramelteon, Suvorexant, Temazepam, Triazolam, Zaleplon, Zolpidem.

                  **Rule C**: The patient has insomnia if prescribed any of the following secondary insomnia medications and reports any symptoms from Definition 1 (difficulty sleeping) or Definition 2 (daytime impairment): Acamprosate, Alprazolam, Clonazepam, Clonidine, Diazepam, Diphenhydramine, Doxepin, Gabapentin, Hydroxyzine, Lorazepam, Melatonin, Mirtazapine, Olanzapine, Quetiapine, Trazodone.

                  **Insomnia status**: The patient is considered to have insomnia if they meet the criteria of Rule A, Rule B, or Rule C ("yes" label). If none of these rules are met, the patient's note is labeled as "no".

                """},
                {"role": "user", "content": text}
            ],
            temperature=0,
            max_tokens=2
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        print(f"Error procesando: {e}")
        return "error"

# Batch classification
# subset_df = df_final.head(20).copy()

df_final['prediction'] = df_final['text'].apply(classify_insomnia)

In [ ]:
# Examples for Few-Shot prompting
few_shot_examples = [
    {"text": ''' male patient in fifties prescribed no drugs

Admission Date:  [**2194-12-26**]       Discharge Date:  [**2195-1-9**]

Date of Birth:   [**2136-9-27**]       Sex:  M

Service:

ADMITTING DIAGNOSIS:  Peritonitis, anastomotic leak, acute
abdomen.

DISCHARGE DIAGNOSIS:
1.  Status post exploratory laparotomy.
2.  Lysis of adhesions.
3.  Diverting ostomy and repair of anastomotic leak.

HISTORY OF PRESENT ILLNESS:  Mr. [**Known lastname 35654**] is a 58-year-old
man with multiple polyps in his colon who underwent a total
abdominal colectomy with ileorectal anastomosis on [**12-8**].
Initial postoperative course was complicated by bleeding
requiring urgent re-exploration on the night of surgery.  He
was discharged to home two days prior to admission tolerating
regular diet, however, upon returning to his home he began to
have crampy abdominal pain, did not pass any flatus or bowel
movements and returned to the Emergency Room with distention.
He denies nausea, vomiting, but did have fevers and no
chills.

PAST MEDICAL HISTORY:  Significant for [**First Name9 (NamePattern2) 7816**] [**Location (un) **]
syndrome, multiple polyps in the colon, peptic ulcer disease,
alcohol abuse.

PAST SURGICAL HISTORY:  As above.

ALLERGIES:  Aspirin ""makes him sick"".

MEDICATIONS:  Loperamide, Protonix, Roxicet and pancreatic
enzymes.

PHYSICAL EXAMINATION:  On admission, significant for
temperature of 101.2, pulse 148, pressure of 102/68.  Heart
rate was regular.  Lungs were clear.  Abdomen was severely
distended with palpable loops of small bowel.  There was
guarding and rebound, left greater than right lower quadrant,
no shakes, no palpable hernias.  Rectal was heme negative.

LABORATORY DATA:  Significant for white count of 45, LFTs
were normal.  Amylase and lipase were normal.  Coags were
normal.  Abdominal x-ray showed free air into the diaphragm
and dilated loops of small bowel.

HOSPITAL COURSE:  The patient was admitted to the operating
room where he underwent an exploratory laparotomy, lysis of
adhesions, ostomy repair of an anastomotic leak.  He was
taken from the operating room to the SICU where he remained
intubated overnight.  He was extubated without difficulty the
following morning, remained in the SICU for observation and
was transferred out to the floor on postoperative day #2.
Once on the floor, he was started on TPN.  Physical therapy
began to follow the patient.  His course was relatively
uncomplicated on the floor.  He had delayed ostomy function
and was maintained on TPN with npo status on Fluconazole,
Flagyl and Levofloxacin for antibiotics.  His intra-abdominal
cultures grew out organisms consistent with fecal soilage of
the peritoneal cavity.  Blood and urine cultures remained
negative throughout his stay.  Once on the floor he had
really no acute issues.  His bowels, as I mentioned, were
slow to return to function, however, once ileostomy started
to work he had no interruption in function.  His diet was
advanced slowly to sips, then clears, then a regular diet
which he was tolerating well by the time of discharge.  He
continued to work physical therapy, was found to be quite
weak, however, was able to ambulate about the floor without
difficulty.  He received ostomy training and became
comfortable, taking care of his ileostomy.  He is being
discharged on postoperative day #15.  His antibiotics had
been off for 24 hours and he remained afebrile.  He is being
discharged with his staples and will follow-up with Dr.
[**Last Name (STitle) **] in approximately one weeks time for staple
removal.

Of note, the patient was on a Somatostatin drip for
approximately a week to slow down his NG tube output,
however, once his ileostomy started to consistently function,
he continued with elevated NG output.  KUB confirmed
placement of the NG tube in the stomach and it was decided
that clamping trials would be initiated.  His residuals after
four hours of clamping were 55 and 10 and NG tube was removed
without difficulty.  The patient does have significant reflux
disease and intermittently had trouble with his reflex,
however, never had any evidence of bowel obstruction and
tolerated his diet throughout.  He is being discharged to
home on Protonix, Imodium prn and Tylenol for pain medication
as well as his preoperative medications.  He will receive
visiting nurse services.




                          [**First Name11 (Name Pattern1) **] [**Last Name (NamePattern4) **], M.D.  [**MD Number(1) 11126**]

Dictated By:[**Last Name (NamePattern1) 22884**]
MEDQUIST36

D:  [**2195-1-9**]  14:30
T:  [**2195-1-9**]  15:18
JOB#:  [**Job Number 35655**]
" ''', "label": "no"},

  {"text": ''' "male patient in seventies prescribed Isosorbide Mononitrate (Extended Release), Lisinopril, Acetaminophen, Warfarin, Prochlorperazine, Morphine Sulfate, Hydromorphone, Magnesium Sulfate, Phytonadione, Sodium Chloride 0.9% Flush, Bisacodyl, Lactulose, Pantoprazole, Psyllium Wafer, Metoprolol, Sevelamer, Senna, Milk of Magnesia, Furosemide, Pantoprazole Sodium, Vial, Sucralfate, Heparin Sodium, D5W, Aspirin EC, Docusate Sodium, Potassium Chloride, Cephalexin, Amlodipine, Isosorbide Dinitrate, Hydralazine HCl, 1/2 NS, Amaryl, Epoetin Alfa, Miralax, Insulin, Atorvastatin, Zolpidem Tartrate, Polysaccharide Iron Complex, Ferrous Sulfate, Lidocaine Jelly 2% (Urojet), Finasteride, Nephrocaps, Glimepiride, Oxycodone-Acetaminophen, Lorazepam Admission Date: [**2161-4-6**] Discharge Date: [**2161-5-1**] Date of Birth: [**2090-12-24**] Sex: M Service: MEDICINE Allergies: Univasc Attending:[**First Name3 (LF) 613**] Chief Complaint: atrial flutter for scheduled ablation Major Surgical or Invasive Procedure: Electrophysiology study with ablation. Tunnelled right internal jugular hemodialysis catheter placement. Hemodialysis EGD History of Present Illness: 70 M with DM, HTN, hyperlipid, DM2, CRF, stroke x 3 s/p R CEA, SAH s/p LMCA aneurysm clip, CAD, LV dysfxn who had p/w CHF, atrial flutter and found on TEE 6 weeks ago to have left atrial appendage thrombus since rate controlled with metoprolol and anticoagulated with warfarin, now returning for flutter ablation. . The atrial flutter was diagnosed when the patient reported palpitations to his visiting nurse. [**First Name (Titles) **] [**Last Name (Titles) 28085**], EKG documented atrial flutter and TEE documented clot in a left atrium appendage. . The patient was recently hospitalized for TIA and had non-invasive carotid studies showing 40% stenosis of [**Country **] and significant plaque in the distal [**Doctor First Name 3098**]. He has reported some persistent numbness in the thighs, incidentally. . He has had primary symptoms of fatigue and dyspnea. He was started on replacement therapy for iron-deficiency anemia. He was diuresed with lasix for lower extremity edema and has had improved symptoms but a rising creatinine, such that on [**2161-3-30**], his was 5.3 (baseline ~mid-3's). He was instructed to hold his lasix, permitting his weight to go increase, and then resume lasix at 60 mg daily. Off lasix3 days later, however, his BP increased to approx 180/110, HR was about 110 and weight up to 180 lbs by remote monitoring, although he denied SOB or CP. He did respond well to lasix 60 mg daily with BP down to 170/74, HR 108, improved symptoms with residual bibasilar crackles. . On presentation, the patient denies dyspnea or chest pain and notes that his exercise capacity is limited more by claudication symptoms in the quadriceps than by DOE. Denies orthopnea or PND. Occasional palpitations but no lightheadedness, dizziness, or vertigo. +Constipation without n/v. +Insomnia only partially explained per patient by nocturia. No pruritis, sleep-wake reversal. Past Medical History: 1. Stroke in [**2145**], ? new stroke in [**5-25**] with decreased word finding ability, Repeat CT stable, EEG nl. [**8-25**] carotid U/S-->80-99% rt carotid stenosis, 50% on left. [**10-25**] right CEA. 2. Subarachnoid hemorrhage in [**2137**] status post middle cerebral artery aneurysm clipping with residual large area of infarct and encephalomalacia 3. Coronary artery disease -[**2130**] MI.[**2143**] CABG at [**Hospital1 2025**], details unavailable followed by [**Name (NI) **] PTCA. [**2149**] cath-->occlusion of all grafts. Repeat CABG NEDH, SVG-->OM1, SVG-->D1, SVG-->RCA. [**2-20**] rest pain, cath-->occluded native RCA and LAD, grafts patent. [**Month/Year (2) 8714**] stented. [**9-25**] routine ETT/[**Doctor Last Name **]--LAD and PDA distribution ischemia on [**Doctor Last Name **]. Cath-->stent of SVG to PDA. -[**1-24**] TTE with EF 30-40% (see below) 4. Hypercholesterolemia 5. Type 2. Diabetes mellitus - no neurologic/opthalmalogic complications. 6. Chronic renal insufficiency - [**4-22**] incr creat 2.4. D/c Univasc, repeat labs-->creat 2.6. Renal U/S nl. [**6-22**] eval Dr. [**Last Name (STitle) 1366**] felt c/w microvascular disease +/- atheroembolic complications post cath. SPEP, UPEP nl. Began Diovan. [**12-27**] incr creat 3.2 persists post cath despite d/c Diovan. 7. Gastroesophageal reflux disease 8. Status post bilateral cataract surgery 9. Hearing loss 10. Peripheral vascular disease with claudication 11. Carpal tunnel syndrome Social History: Married, lives with wife. Former accountant, retired in [**2145**]. Former tobacco smoker, quit in [**2144**]. Social alcohol use. Denies drug use. Family History: Father with strokes. Brother with coronary artery disease. Physical Exam: T: 96.7F, BP: R-168/90 L-148/90, P: 68, R: 20, SaO2:96%,RA NAD, nondiaphoretic Edentulous, no OP lesions, no scleral/sublingual icterus. No LAD, Carotids 2+ without bruits, JVP 8cm H20 Chest with rales confined to bases. Heart with irregularly irregular rate. No S3,S4 heard consistently. No M/R. +BS, quite distended but nontender. No HSM by percussion or palpation. 2+ left femoral and dp. 1+ right femoral and dp. Trace left and 1+ right leg edema. 1-second capillary refill. Neuro A/Ox3 with word-finding difficulties, occasionally stuttering, motor with 5/5 throughout except 4+/5 RLE and [**3-26**] LLE. (+)R Babinski sign. (-)L Babinski sign. Pertinent Results: . LABS [**3-16**]: WBC 7.4, HCT 26.0-->28.5, PLT 323 [**3-19**]: PT 23.0, INR 3.3 [**3-27**]: NA 135, K 4.4, CL 92, CO2 29, BUN 87, CR 5.3 (baseline ~3.3), GAP=14 [**3-15**]: CK 28, TrT 0.09 [**3-16**]: Fe 29, TIBC 307, [**Last Name (un) **] 118, TRF 236 [**3-15**]: Chol 71, TG 126, HDL 25, [**3-14**]: HbA1c 8.1 [**3-14**]: Dig 0.7 . [**2161-4-6**] 05:08PM WBC-9.1 HCT-27.7* MCV-81* PLT COUNT-144*# [**2161-4-6**] 05:08PM PT-17.3* PTT-33.4 INR(PT)-2.0 [**2161-4-6**] 05:08PM SODIUM-140 POTASSIUM-3.5 CHLORIDE-95* TOTAL CO2-28 UREA N-99* CREAT-6.2* GLUCOSE-146* ANION GAP-17* ALBUMIN-3.9 MAGNESIUM-3.0* . [**2161-4-6**] 08:16PM URINE UREA N-536 CREAT-53 SODIUM-49 CHLORIDE-42 TOT PROT-117 PROT/CREA-2.2* . . . TTE, [**2161-1-22**] The left atrium is moderately dilated. The right atrium is moderately dilated. No atrial septal defect is seen by 2D or color Doppler. There is mild symmetric left ventricular hypertrophy. The left ventricular cavity size is normal. Overall left ventricular systolic function is moderately depressed (ejection fraction 30-40 percent) with global hypokinesis that may be somewhat worse in the inferior and posterior walls. No masses or thrombi are seen in the left ventricle. There is no ventricular septal defect. Right ventricular chamber size and free wall motion are normal. The ascending aorta is mildly dilated. The aortic arch is mildly dilated. There are focal calcifications in the aortic arch. The aortic valve leaflets (3) are mildly thickened but aortic stenosis is not present. No aortic regurgitation is seen. The mitral valve leaflets are mildly thickened. There is no mitral valve prolapse. Moderate (2+) mitral regurgitation is seen. There is no pericardial effusion. . TEE, [**2161-2-13**] 1. The left atrium is dilated. A definite thrombus is seen in the left atrial appendage. It seem well organised and has a small stalk to which it is attached. 2. A patent foramen ovale is present. 3. The left ventricular cavity size is normal. LV systolic function appears depressed. 4. There are complex (>4mm and/or mobile) atheroma in the ascending aorta, aortic arch, and in the descending thoracic aorta. 5. The aortic valve leaflets (3) are mildly thickened. Trace aortic regurgitation is seen. 6. The mitral valve leaflets are mildly thickened. Mild (1+) mitral regurgitation is seen. . CARDIAC CATHETERIZATION, [**2160-11-3**]: FINAL DIAGNOSIS: 1. Three vessel coronary artery disease. 2. Normal ventricular function. 3. Systolic hypertension 4. Successful stenting of the retrograde limb of the PDA via the SVG to the PDA with two overlapping Drug Eluting Stents. COMMENTS: 1. Coronary arteriography revealed a right dominant system with severe native three vessel disease. The LMCA was diffusely diseased. The LAD was totally occluded after a small D1 branch. The distal vessel filled well via a patent LIMA. The SVG to D2 graft was not visualized (aortography was performed) and is presumed to be occluded. The [**Month/Day/Year **] was totally occluded after the first OM, which was small. A large branching OM2 was well filled by a patent SVG. The RCA was totally occluded in its mid segment. The SVG to PDA was widely patent however the retrograde limb of the PDA had a 90% stenosis and the antegrade limb had a 50% stenosis. 2. Limited hemodynamics revealed systolic hypertension. 3. Left ventriculography was not performed due to concerns about the patient's renal function. Ascending aortography was performed to assess location of the bypass grafts. 4. Successful stenting of the retrograde limb of the PDA via the SVG to the PDA with two overlapping DES, a distal 3.0x23mm Cypher DES and a more proximal 3.5x13mm Cypher [**Name Prefix (Prefixes) **] [**Last Name (Prefixes) 7930**] to 3.5mm at the mid/proximal segment. Brief Hospital Course: A/P: 70 M with type II diabetes mellitus, hypertension, hyperlipidemia, chronic renal insufficiency, stroke x 3 s/p R CEA, SAH s/p LMCA aneurysm clip, coronary artery disease with LV dysfunction who had been admitted with decompensated heart failure, atrial flutter and found on TEE 6 weeks ago to have left atrial appendage thrombus. Since that time he has been rate controlled with metoprolol and anticoagulated with warfarin. He was admitted for flutter ablation. Hospital course was complicated. During his hospitalization the following problems were addressed: 1. Atrial flutter: Patient was treated with heparin gtt and underwent aflutter ablation [**2161-4-7**]. Coumadin was subsequently restarted. His INR became supratherapeutic, and he developed bilateral retroperitoneal bleeds. All anticoagulation was held, and he was transfused PRBC to maintain Hct. As the RP bleeds did not improve in size, FFP was administered. The patient developed a transfusion reaction to the FFP that was treated with solumedrol and benadryl. He was admitted overnight to the CCU for close monitoring. Symptoms resolved, and he was transferred back to the floor. Subsequent plans to restart anticoagulation were postponed as his Hct continued to drift down, requiring further PRBC transfusions. A GI consult was obtained and he had an EGD which revealed gastritis, which was cauterized. His hematocrit was stable thereafter. He remained in sinus rhythm. He was restarted on anticoagulation given his [**Name Prefix (Prefixes) **] [**Last Name (Prefixes) 7388**] (and needed for 6 weeks post ablation in any case) and his goal INR is 2.0-2.5. Please be very careful in preventing overcoagulation. 2. CHF: Patient has congestive heart failure with some degree of brittle volume status. He was treated initially with nitrates, beta-blockade, and hydralazine for afterload reduction, and diuresed with lasix as needed. Eventually, the hydralazine was discontinued, and ACE-inhibitor restarted after initiation of hemodialysis and no further concern for renal disease. Volume status stabilized, and he was continued on metoprolol and ACE-I for secondary prevention. 3. CAD: Continued isordil, metoprolol, aspirin and ACEI for secondary prevention. There were no acute issues. 4.Chronic kidney disease: Patient has a history of diabetic and hypertensive nephropathy with volume-dependent kidney function and baseline creatinine in mid-3 range with creatinine clearnace of ~30 at optimum. On presentation, creatinine was >6 with increased anion gap. Nephrology service was consulted, and he was started on hemodialysis. A tunnelled dialysis line was placed by IR, and he was started on Mon/Wed/Fri dialysis. He eventually had an AF graft placed which was used for his next HD session without problems. The Renal team recommended keeping the tunnelled line in place until the AV graft has been consistently functional. 5. Type II diabetes mellitus: Oral hypoglycemics were held, and he was treated with a sliding insulin scale. Recent HgbA1c was 8.1. 6. TRALI: Pt recieved FFP during when he had his RP bleed and developed fevers, SOB and infiltrates. Found to have TRALI and treated with IV steroids X 1 day. Did not require intubation. 7. L femoral compressive neuropathy: developed as a complication of his retroperitoneal bleeding, causing significant LE pain and weakness. He slowly improved with PT and analgesics; he will continue aggressive PT at rehab. Medications on Admission: MEDS AT HOME: AMARYL 2MG--2 qam, one every evening AMBIEN 5MG--One by mouth at bedtime as needed ASPIRIN 325MG--One every day EPOETIN ALFA 2,000 unit/mL--1 cc ([**2155**] u) sc three times weekly HYDRALAZINE HCL 25 mg--1 tablet(s) by mouth three times a day ISOSORBIDE DINITRATE 30 mg--1 three times a day LASIX 40 mg--2 tablet(s) by mouth once a day LIPITOR 10MG--One by mouth every day NIFEREX 60 mg--1 capsule(s) by mouth once a day METROPOLOL 100mg TID WARFARIN SODIUM 3 mg--as directed (held 3 days prior to admission) WARFARIN SODIUM 5 mg--as directed (held 3 days prior to admission) Milk of magnesium Colace Senna Discharge Medications: 1. Zolpidem Tartrate 5 mg Tablet Sig: One (1) Tablet PO HS (at bedtime) as needed for prn insomnia. Disp:*10 Tablet(s)* Refills:*0* 2. Finasteride 5 mg Tablet Sig: One (1) Tablet PO DAILY (Daily). Disp:*30 Tablet(s)* Refills:*2* 3. Atorvastatin Calcium 80 mg Tablet Sig: One (1) Tablet PO DAILY (Daily). Disp:*30 Tablet(s)* Refills:*2* 4. Sevelamer HCl 800 mg Tablet Sig: Two (2) Tablet PO TID (3 times a day). Disp:*180 Tablet(s)* Refills:*2* 5. Isosorbide Mononitrate 30 mg Tablet Sustained Release 24HR Sig: One (1) Tablet Sustained Release 24HR PO DAILY (Daily). Disp:*30 Tablet Sustained Release 24HR(s)* Refills:*2* 6. Lisinopril 20 mg Tablet Sig: Two (2) Tablet PO DAILY (Daily). Disp:*60 Tablet(s)* Refills:*2* 7. Metoprolol Tartrate 50 mg Tablet Sig: One (1) Tablet PO BID (2 times a day). Disp:*60 Tablet(s)* Refills:*2* 8. Pantoprazole Sodium 40 mg Tablet, Delayed Release (E.C.) Sig: One (1) Tablet, Delayed Release (E.C.) PO Q24H (every 24 hours). Disp:*30 Tablet, Delayed Release (E.C.)(s)* Refills:*2* 9. Aspirin 81 mg Tablet, Delayed Release (E.C.) Sig: One (1) Tablet, Delayed Release (E.C.) PO DAILY (Daily). Disp:*30 Tablet, Delayed Release (E.C.)(s)* Refills:*2* 10. Warfarin Sodium 2.5 mg Tablet Sig: One (1) Tablet PO HS (at bedtime). Disp:*30 Tablet(s)* Refills:*2* 11. Heparin Sod (Porcine) in D5W 100 unit/mL Parenteral Solution Sig: One (1) Intravenous ASDIR (AS DIRECTED): until INR >2.0 . Disp:*qs * Refills:*2* Discharge Disposition: Extended Care Facility: [**Hospital3 1107**] [**Hospital **] Hospital - [**Location (un) 38**] Discharge Diagnosis: Atrial Flutter Congestive heart failure Acute renal failure Trali Retroperitoneal Bleed GI Bleed left femoral compressive neuropathy Discharge Condition: stable and improved Discharge Instructions: Please call your physician with any new, worsening, or different shortness of breath, chest pain, lightheadedness, fatigue, nausea, vomiting, or confusion. Please continue your current medications as previously directed. Please follow up in dialysis as per the recommendations of the nephrology social worker, [**Name (NI) **] [**Name (NI) 17926**]. Followup Instructions: Please note the following previously scheduled appointments: Provider: [**First Name11 (Name Pattern1) 1877**] [**Last Name (NamePattern1) 1878**], M.D. Where: [**Hospital6 29**] MEDICAL SPECIALTIES Phone:[**Telephone/Fax (1) 435**] Date/Time:[**2161-5-28**] 2:30 Provider: [**Name10 (NameIs) 251**] [**Last Name (NamePattern4) 252**], M.D. Where: [**Hospital6 29**] [**Hospital3 1935**] CENTER Phone:[**Telephone/Fax (1) 253**] Date/Time:[**2161-10-7**] 9:30 Provider: [**Name10 (NameIs) **] [**Name11 (NameIs) 3627**] [**Name12 (NameIs) 3628**] VASCULAR [**Name12 (NameIs) 3628**] Where: VASCULAR [**Name12 (NameIs) 3628**] Date/Time:[**2162-3-29**] 10:00 [**First Name5 (NamePattern1) **] [**Last Name (NamePattern1) **] MD [**MD Number(2) 617**] Completed by:[**2161-5-1**]" ''', "label": "yes"}
]

In [ ]:
def classify_insomnia_few_shot(text):
    # Building the prompt using the examples and the self-check strategy
    messages = [
        {"role": "system", "content": """You are a medical classifier. Respond ONLY with the word 'yes' if the text indicates insomnia, or 'no' if it does not. Do not explain. The following are the rules to decide your response:

            **Definition 1: Difficulty Sleeping**
            The patient is considered to have difficulty sleeping if they report any of the following:
            1. Trouble initiating sleep.
            2. Trouble maintaining sleep.
            3. Waking up earlier than desired.
            4. An explicit mention of insomnia.

            **Definition 2: Daytime Impairment**
            The patient is considered to have daytime impairment if they report any of the following:
            1. Fatigue or malaise.
            2. Impaired attention, concentration, or memory.
            3. Impaired social, family, occupational, or academic performance.
            4. Mood disturbance or irritability.
            5. Daytime sleepiness.
            6. Behavioral problems such as hyperactivity, impulsivity, or aggression.
            7. Decreased motivation, energy, or initiative.
            8. Proneness to errors or accidents.
            9. Concerns or dissatisfaction with sleep.
            10. An explicit mention of insomnia.

            **Rule A:** The patient is considered to have insomnia if they meet both Definition 1 (difficulty sleeping) and Definition 2 (daytime impairment).

            **Rule B**: The patient has insomnia if prescribed any of the following primary insomnia medications: Estazolam, Eszopiclone, Flurazepam, Lemborexant, Quazepam, Ramelteon, Suvorexant, Temazepam, Triazolam, Zaleplon, Zolpidem.

            **Rule C**: The patient has insomnia if prescribed any of the following secondary insomnia medications and reports any symptoms from Definition 1 (difficulty sleeping) or Definition 2 (daytime impairment): Acamprosate, Alprazolam, Clonazepam, Clonidine, Diazepam, Diphenhydramine, Doxepin, Gabapentin, Hydroxyzine, Lorazepam, Melatonin, Mirtazapine, Olanzapine, Quetiapine, Trazodone.

            **Insomnia status**: The patient is considered to have insomnia if they meet the criteria of Rule A, Rule B, or Rule C ("yes" label). If none of these rules are met, the patient's note is labeled as "no".

        """}
    ]

    # Adding the examples to the conversation history (Few-Shot)
    for example in few_shot_examples:
        messages.append({"role": "user", "content": example["text"]})
        messages.append({"role": "assistant", "content": example["label"]})

    # Adding the actual text to classify
    messages.append({"role": "user", "content": text})

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0,
            max_tokens=2 # Increase to 2 in case the tokenizer separates the 'yes'/'no' from a space.
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        print(f"Error: {e}")
        return "error"

# Batch Processing for Training - Very Large Corpus

In [ ]:
# Run the classification (using a subset if the corpus is very large in order to save tokens)

# Example using the first 20 records for quick testing:
#subset_df = df_final.head(40).copy()
#subset_df = df_final.iloc[130:].copy()

# Example using the last 20 records for quick testing:
#subset_df = df_final.tail(20).copy()

#df_final['prediction'] = subset_df['text'].apply(classify_insomnia_few_shot)

In [ ]:
predicciones1_40 = df_final["prediction"].iloc[0:40]
#df_final.iloc[130:]

In [ ]:
df_final['prediction'] = predicciones
df_final

In [ ]:
# Save prediction results for the training dataset
df_final.to_csv('predictions_train.csv', index=False)

## Filter the df_final results dataframe to display only errors - Training

In [ ]:
df_final_errores = df_final[df_final['label'] != df_final['prediction']]
df_final_errores

,note_id,text,label,prediction
2,1272946,"female patient in sixties prescribed D5W, D5 1...",yes,no
7,1320608,female patient in hundreds prescribed Clopidog...,yes,no
9,1328271,"male patient in thirties prescribed Diazepam, ...",no,yes
12,1334025,female patient in thirties prescribed Ferrous ...,yes,no
15,25898,"male patient in fifties prescribed NS, Sodium ...",no,yes
21,58969,female patient in sixties prescribed no drugs\...,no,yes


In [ ]:
df_final_errores.shape

(6, 4)

In [ ]:
# Save incorrectly predicted results for the training dataset
df_final_errores.to_csv('failed_predictions_train.csv', index=False)

## 4. Metric Evaluation on the Training Set

We use scikit-learn to compute the model performance against the ground-truth labels.

In [ ]:
# Filter potential API errors to avoid compromising the calculation
df_eval = df_final[df_final['prediction'].isin(['yes', 'no'])]

y_true = df_eval['label']
y_pred = df_eval['prediction']

precision = precision_score(y_true, y_pred, pos_label='yes')
recall = recall_score(y_true, y_pred, pos_label='yes')
f1 = f1_score(y_true, y_pred, pos_label='yes')

print("--- METRICS ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\nComplete report:")
print(classification_report(y_true, y_pred))

--- MÉTRICAS DE CLASIFICACIÓN ---
Precision: 0.9231
Recall:    0.9000
F1-Score:  0.9114

Reporte Completo:
              precision    recall  f1-score   support

          no       0.90      0.92      0.91        76
         yes       0.92      0.90      0.91        80

    accuracy                           0.91       156
   macro avg       0.91      0.91      0.91       156
weighted avg       0.91      0.91      0.91       156



## Convert to JSON and Save for Submission - Training

In [ ]:
import pandas as pd
import json

# Path to your CSV file in Drive
csv_path = "predictions_training.csv"

# Path where the JSON file will be saved
json_path = "predictions_training.json"

# Read CSV
df = pd.read_csv(csv_path)

# Create dictionary in the required format
output = {
    str(row["note_id"]): {
        "Insomnia": str(row["prediction"]).lower()
    }
    for _, row in df.iterrows()
}

# Save as JSON
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=4, ensure_ascii=False)

print(f"File saved in JSON at: {json_path}")

Archivo guardado en formato JSON en: /content/drive/MyDrive/Profesor Ocasional Tiempo Completo UdeA/Investigación/Trabajos para publicaciones y competencias/SMM4H 2026/Insomnia Task/predictions/validation/predictions_validation_self_check_Task_subtask_1.json


# Validation

Complete processing for the validation set, consisting of only 23 clinical records.

In [ ]:
# Complete dataset
df_final['prediction'] = df_final['text'].apply(classify_insomnia_few_shot)

Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-hcr9xttpfQ1y5hWoIIwI1ra8 on tokens per min (TPM): Limit 200000, Used 194408, Requested 6212. Please try again in 186ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}


In [ ]:
df_final.head(30)

,note_id,text,label,prediction
0,10686,male patient in forties prescribed no drugs\n\...,no,no
1,1262007,female patient in forties prescribed Iso-Osmot...,yes,yes
2,1272946,"female patient in sixties prescribed D5W, D5 1...",yes,no
3,1279443,female patient in eighties prescribed no drugs...,yes,yes
4,1281746,male patient in fifties prescribed Iso-Osmotic...,no,no
5,1284059,female patient in fifties prescribed Phytonadi...,yes,yes
6,1288748,"male patient in forties prescribed D5W, Midazo...",yes,yes
7,1320608,female patient in hundreds prescribed Clopidog...,yes,yes
8,1324558,"male patient in seventies prescribed D5 1/2NS,...",no,no
9,1328271,"male patient in thirties prescribed Diazepam, ...",no,yes


In [ ]:
# Save prediction results for the validation dataset
df_final.to_csv('predictions_validation.csv', index=False)

## Filter the results dataframe to display errors

In [ ]:
df_final_errores = df_final[df_final['label'] != df_final['prediction']]
df_final_errores

,note_id,text,label,prediction
2,1272946,"female patient in sixties prescribed D5W, D5 1...",yes,no
9,1328271,"male patient in thirties prescribed Diazepam, ...",no,yes
12,1334025,female patient in thirties prescribed Ferrous ...,yes,no
18,56587,male patient in seventies prescribed no drugs\...,yes,no
19,56803,female patient in thirties prescribed Albumin ...,no,error
21,58969,female patient in sixties prescribed no drugs\...,no,yes


In [ ]:
df_final_errores.shape

(6, 4)

In [ ]:
# Save incorrectly predicted results for the training dataset
df_final_errores.to_csv('failed_predictions_validation.csv', index=False)

## Metric Evaluation on the Validation Set

In [ ]:
# Filter potential API errors to avoid compromising the calculation
df_eval = df_final[df_final['prediction'].isin(['yes', 'no'])]

y_true = df_eval['label']
y_pred = df_eval['prediction']

precision = precision_score(y_true, y_pred, pos_label='yes')
recall = recall_score(y_true, y_pred, pos_label='yes')
f1 = f1_score(y_true, y_pred, pos_label='yes')

print("--- CLASSIFICATION METRICS ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\nComplete Report:")
print(classification_report(y_true, y_pred))

--- MÉTRICAS DE CLASIFICACIÓN ---
Precision: 0.7778
Recall:    0.7000
F1-Score:  0.7368

Reporte Completo:
              precision    recall  f1-score   support

          no       0.77      0.83      0.80        12
         yes       0.78      0.70      0.74        10

    accuracy                           0.77        22
   macro avg       0.77      0.77      0.77        22
weighted avg       0.77      0.77      0.77        22



## Convert to JSON and Save for Submission - Validation

In [ ]:
import pandas as pd
import json

# Path to your CSV file in Drive
csv_path = "predictions_validation.csv"

# Path where the JSON file will be saved
json_path = "validation/subtask_1.json"

# Read CSV
df = pd.read_csv(csv_path)

# Create dictionary in the required format
output = {
    str(row["note_id"]): {
        "Insomnia": str(row["prediction"]).lower()
    }
    for _, row in df.iterrows()
}

# Save as JSON
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=4, ensure_ascii=False)

print(f"File saved in JSON at: {json_path}")

Archivo guardado en formato JSON en: /content/drive/MyDrive/Profesor Ocasional Tiempo Completo UdeA/Investigación/Trabajos para publicaciones y competencias/SMM4H 2026/Insomnia Task/predictions/validation/subtask_1.json


# 5. Inference for New Texts

Here is the function for testing arbitrary texts directly within the notebook.

In [ ]:
def predict_new_text(input_string):
    prediction = classify_insomnia(input_string)
    print(f"Text: {input_string[:50]}...")
    print(f"Insomnia?: **{prediction.upper()}**")
    return prediction

# Usage example:
nuevo_texto = "..."
predict_new_text(nuevo_texto)

Texto: ......
¿Tiene Insomnio?: **NO**


'no'